In [1]:
pip install mujoco gymnasium stable-baselines3[extra] sympy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import gymnasium as gym
import stable_baselines3
from stable_baselines3 import PPO
from stable_baselines3 import SAC, TD3
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.vec_env import SubprocVecEnv
import Spacecraft_V1

**Optinal:** This section is only for checking useable cpu cores

In [8]:
pip install os psutil

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement os (from versions: none)

[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for os


In [2]:
import os
import psutil # Install via: pip install psutil

logical_cores = os.cpu_count()
physical_cores = psutil.cpu_count(logical=False)

print(f"Logical cores (threads): {logical_cores}")
print(f"Physical cores: {physical_cores}")

Logical cores (threads): 16
Physical cores: 8


**Declare your Vectorized Env**

In [ ]:
if __name__ == '__main__':
    gym.register(
        id="Spacecraft_V1",
        entry_point="Spacecraft_V1:SpacecraftEnv",
        max_episode_steps=2000,
    )
    # Create a MuJoCo environment
    env = gym.make("Spacecraft_V1", render_mode="cpu")

    vec_env = make_vec_env(
            env,
            n_envs=4,
            vec_env_cls=SubprocVecEnv, 
        )
    
    # Instantiate the PPO model
    model = PPO("MlpPolicy", vec_env, verbose=1) 
    model.learn(total_timesteps=500000)       


c:\Users\texei\OneDrive\WindowsCode\EGR444_final\.venv\Lib\site-packages\gymnasium\envs\registration.py:732: UserWarning: WARN: The environment is being initialised with render_mode='cpu' that is not in the possible render_modes (['human', 'rgb_array', 'depth_array']).
  logger.warn(


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [4]:
# Train the model
model.learn(total_timesteps=500000)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 28.3     |
|    ep_rew_mean     | -445     |
| time/              |          |
|    fps             | 719      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 28.9        |
|    ep_rew_mean          | -423        |
| time/                   |             |
|    fps                  | 504         |
|    iterations           | 2           |
|    time_elapsed         | 8           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.010333389 |
|    clip_fraction        | 0.124       |
|    clip_range           | 0.2         |
|    entropy_loss         | -12.8       |
|    explained_variance   | 0.00236     |
|    learning_rate        | 0.

In [5]:
# Save the model
model.save("ppo_spacecraft")

**Evaluate model**

In [6]:
model = PPO.load("ppo_spacecraft")

eval_env = gym.make("Spacecraft_V1", render_mode="human")
obs, info = eval_env.reset()
for _ in range(2000):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    eval_env.render()
    if terminated or truncated:
        obs, info = eval_env.reset()


**SAC and TD3** (run the registration cell above first so `Spacecraft_V1` exists)

In [7]:
# SAC: explores through its own entropy term, so no action noise needed
from stable_baselines3 import SAC

sac_env = gym.make("Spacecraft_V1")        # no render_mode while training
sac_model = SAC("MlpPolicy", sac_env, verbose=1, device="cpu")
sac_model.learn(total_timesteps=200000)
sac_model.save("sac_spacecraft")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50.2     |
|    ep_rew_mean     | -451     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 130      |
|    time_elapsed    | 1        |
|    total_timesteps | 201      |
| train/             |          |
|    actor_loss      | -10.5    |
|    critic_loss     | 363      |
|    ent_coef        | 0.971    |
|    ent_coef_loss   | -0.445   |
|    learning_rate   | 0.0003   |
|    n_updates       | 100      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 36       |
|    ep_rew_mean     | -470     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 105      |
|    time_elapsed    | 2        |
|    total_timesteps | 288      |
| train/             |

KeyboardInterrupt: 

In [ ]:
# TD3: needs action noise, otherwise it only explores for the first 100 steps
import numpy as np
from stable_baselines3 import TD3
#This is because TD3 will act deterministically after the first 100 steps, so we need to add noise to the actions to encourage exploration
from stable_baselines3.common.noise import NormalActionNoise

td3_env = gym.make("Spacecraft_V1")        # no render_mode while training
n_actions = td3_env.action_space.shape[0]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

td3_model = TD3("MlpPolicy", td3_env, action_noise=action_noise, verbose=1, device="cpu")
td3_model.learn(total_timesteps=200000)
td3_model.save("td3_spacecraft")

In [ ]:
# Watch a SAC or TD3 model (load with the same class you trained with)


model = SAC.load("sac_spacecraft") 
#model = PPO.load("ppo_spacecraft") 
#model = TD3.load("td3_spacecraft") 

eval_env = gym.make("Spacecraft_V1", render_mode="human")
obs, info = eval_env.reset()
for _ in range(3000):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)   # human mode already renders
    if terminated or truncated:
        obs, info = eval_env.reset()
eval_env.close()